# NOVA Blender Skeleton Motion v32.3 — DUO FOOT LOCK
Настоящий Blender control: два скелета, IK, foot-lock и contact-check поверх видео.


In [ ]:
# NOVA Blender Skeleton Motion v32.3 — DUO FOOT LOCK
REFERENCE_DURATION=5.0; CONTROL_FPS=30; OUTPUT_WIDTH=432; OUTPUT_HEIGHT=768
import os,sys,json,shutil,subprocess
from pathlib import Path
from google.colab import files
from IPython.display import Video,FileLink,display
env=os.environ.copy(); env['DEBIAN_FRONTEND']='noninteractive'
subprocess.run(['sudo','apt-get','update','-qq'],check=True,env=env)
subprocess.run(['sudo','apt-get','install','-y','--no-install-recommends','blender','ffmpeg'],check=True,env=env)
repo=Path('/content/nova'); branch='blender-colab-studio'
if (repo/'.git').exists():
    subprocess.run(['git','-C',str(repo),'fetch','origin',branch],check=True); subprocess.run(['git','-C',str(repo),'checkout',branch],check=True); subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch',branch,'https://github.com/magomedt149/nova-robot.git',str(repo)],check=True)
uploaded=files.upload()
videos=[n for n in uploaded if Path(n).suffix.lower() in {'.mp4','.mov','.mkv','.webm','.avi'}]
jsons=[n for n in uploaded if Path(n).suffix.lower()=='.json']
if not videos: raise RuntimeError('Нужно reference video.')
work=Path('/content/NOVA_DUO_CONTROL'); shutil.rmtree(work,ignore_errors=True); work.mkdir()
src=Path('/content')/Path(videos[0]).name; src.write_bytes(uploaded[videos[0]])
ref=work/'reference_video_9x16_5s.mp4'; motion=work/'NOVA_DUO_WALK_LAUGH_FOOTLOCK_5s.json'; scripts=repo/'blender-colab/scripts'
subprocess.run(['ffmpeg','-y','-i',str(src),'-t','5','-vf',f'scale={OUTPUT_WIDTH}:{OUTPUT_HEIGHT}:force_original_aspect_ratio=decrease,pad={OUTPUT_WIDTH}:{OUTPUT_HEIGHT}:(ow-iw)/2:(oh-ih)/2:black,fps={CONTROL_FPS}','-an','-c:v','libx264','-pix_fmt','yuv420p',str(ref)],check=True)
if jsons:
    motion.write_bytes(uploaded[jsons[0]])
    if int(json.loads(motion.read_text(encoding='utf-8')).get('version',0))<3: raise RuntimeError('JSON старый. В NOVA скачайте Skeleton + Foot Lock v32.3.')
else:
    subprocess.run([sys.executable,str(scripts/'generate_dual_walk_laugh_motion.py'),'--output',str(motion),'--fps','30','--duration','5'],check=True)
subprocess.run(['blender','--background','--python',str(scripts/'make_dual_walk_laugh_control.py'),'--','--motion',str(motion),'--output-dir',str(work),'--width',str(OUTPUT_WIDTH),'--height',str(OUTPUT_HEIGHT)],check=True)
control=work/'nova_duo_skeleton_control.mp4'; check=work/'NOVA_SKELETON_CONTACT_CHECK.mp4'
subprocess.run([sys.executable,str(scripts/'make_contact_check.py'),'--video',str(ref),'--motion',str(motion),'--output',str(check)],check=True)
assert control.exists() and check.exists()
archive=Path(shutil.make_archive('/content/NOVA_DUO_FOOTLOCK_CONTROL_5s','zip',root_dir=work))
print('REFERENCE'); display(Video(str(ref),embed=True,width=280))
print('REAL BLENDER SKELETON'); display(Video(str(control),embed=True,width=280))
print('CONTACT CHECK: green foot = planted'); display(Video(str(check),embed=True,width=280))
display(FileLink(str(archive)))
print('Для Wan/WanGP используйте nova_duo_skeleton_control.mp4. Contact-check — только проверка.')
